In [5]:
import numpy as np
import pandas as pd
import libpysal
from libpysal.weights import Queen
import matplotlib.pyplot as plt
import geopandas as gpd
import scipy as sp

RANDOM_SEED = 5781

In [6]:
def get_spatial_filters_by_cbsa(gdf, target_col, cbsa_col='cbsa', threshold=0.25, max_filters=5):
    """
    Computes spatial eigenvectors for each CBSA independently and stitches them back.

    Extracts Spatial Eigenvectors (filters) to control for spatial autocorrelation.
    
    Args:
        gdf (GeoDataFrame): Data (Census Block Groups).
        target_variable_name (str): The outcome variable column.
        threshold (float): Min Moran's I to keep an eigenvector (default 0.25 captures broad trends).
        
    Returns:
        pd.DataFrame: A set of selected eigenvectors to add to the model.
        
    """
    all_filters = []
    
    # Iterate through each unique CBSA
    unique_cbsas = gdf[cbsa_col].unique()
    print(f"Processing {len(unique_cbsas)} CBSAs...")

    for cbsa in unique_cbsas:
        # Subset data for this CBSA
        subset = gdf[gdf[cbsa_col] == cbsa].copy()
          
        try:
            # 1. Weights for this CBSA subset
            w = Queen.from_dataframe(subset)
            w.transform = 'r'
            
            # 2. Centering Matrix
            n = len(subset)
            I = np.eye(n)
            M = I - np.ones((n, n)) / n
            W_full = w.full()[0]
            MWM = M @ W_full @ M
            
            # 3. Eigen-decomposition
            evals, evecs = np.linalg.eigh(MWM)
            
            # Sort descending
            idx = evals.argsort()[::-1]
            evals, evecs = evals[idx], evecs[:, idx]
            
            # 4. Filter selection (Positive Autocorrelation)
            # We take the top 'max_filters' that meet the Moran's I threshold
            valid_idx = np.where(evals > threshold)[0]
            top_idx = valid_idx[:max_filters]
            
            if len(top_idx) > 0:
                cbsa_filters = pd.DataFrame(
                    evecs[:, top_idx], 
                    index=subset.index,
                    columns=[f'SF_{i}' for i in range(len(top_idx))]
                )
                all_filters.append(cbsa_filters)
                
        except Exception as e:
            print(f"Skipping CBSA {cbsa} due to error: {e}")

    # 5. Stitching
    # Concatenate all CBSA filters. Rows not in a processed CBSA will be NaN.
    # We fill NaNs with 0 (meaning no spatial effect attributed).
    stitched_filters = pd.concat(all_filters).reindex(gdf.index).fillna(0)
    
    return stitched_filters

In [7]:
tran_par_file = '/work/hawkins_lab/vulcan/data/vulcan/parquet/vulcan_ONR_epa_climate.parquet'
tran_df = gpd.read_parquet(tran_par_file)
tran_df = tran_df[(tran_df["value_weig"]>0) & (tran_df["totpop"]>0)]

res_par_file = '/work/hawkins_lab/vulcan/data/vulcan/parquet/vulcan_RES_epa_climate.parquet'
res_df = gpd.read_parquet(res_par_file)
res_df = res_df[(res_df["value_weig"]>0) & (res_df["totpop"]>0)]

elec_par_file = '/work/hawkins_lab/vulcan/data/vulcan/parquet/vulcan_SC2_epa_climate.parquet'
elec_df = gpd.read_parquet(elec_par_file)
elec_df = elec_df[(elec_df["res_tc"]>0) & (elec_df["totpop"]>0)]

In [8]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

for file, df in [(tran_par_file, tran_df), (res_par_file, res_df), (elec_par_file, elec_df)]:
    df['cbsa_eig'] = np.where(df['cbsa'] < 10000, df['statefp'].astype(int), df['cbsa'])
    
    unique_cbsas = df['cbsa'].unique()
    for cbsa in unique_cbsas:
        # Need a minimum number of units to calculate spatial patterns
        if df[df['cbsa'] == cbsa].shape[0] < 100:
            df.loc[df.cbsa==cbsa, 'cbsa_eig'] = df['statefp'].astype(int)
            
    temp = get_spatial_filters_by_cbsa(df, 'value_weig', cbsa_col='cbsa_eig')

    print(temp.describe())

    df = pd.concat([df,temp], axis=1)
    df.to_parquet(file)

Processing 335 CBSAs...


/home/jfhawkin/.local/lib/python3.12/site-packages/libpysal/weights/contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 3 disconnected components.
  W.__init__(self, neighbors, ids=ids, **kw)
/home/jfhawkin/.local/lib/python3.12/site-packages/libpysal/weights/contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 4 disconnected components.
  W.__init__(self, neighbors, ids=ids, **kw)
/home/jfhawkin/.local/lib/python3.12/site-packages/libpysal/weights/contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 4 disconnected components.
 There is 1 island with id: 5337.
  W.__init__(self, neighbors, ids=ids, **kw)


('WARNING: ', 5337, ' is an island (no neighbors)')


/home/jfhawkin/.local/lib/python3.12/site-packages/libpysal/weights/contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 2 disconnected components.
 There is 1 island with id: 309.
  W.__init__(self, neighbors, ids=ids, **kw)


('WARNING: ', 309, ' is an island (no neighbors)')


...

('WARNING: ', 4701, ' is an island (no neighbors)')
('WARNING: ', 9931, ' is an island (no neighbors)')
('WARNING: ', 11568, ' is an island (no neighbors)')
('WARNING: ', 13233, ' is an island (no neighbors)')


/home/jfhawkin/.local/lib/python3.12/site-packages/libpysal/weights/contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 2 disconnected components.
  W.__init__(self, neighbors, ids=ids, **kw)
/home/jfhawkin/.local/lib/python3.12/site-packages/libpysal/weights/contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 5 disconnected components.
  W.__init__(self, neighbors, ids=ids, **kw)
/home/jfhawkin/.local/lib/python3.12/site-packages/libpysal/weights/contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 4 disconnected components.
 There is 1 island with id: 691.
  W.__init__(self, neighbors, ids=ids, **kw)


('WARNING: ', 691, ' is an island (no neighbors)')


/home/jfhawkin/.local/lib/python3.12/site-packages/libpysal/weights/contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 4 disconnected components.
  W.__init__(self, neighbors, ids=ids, **kw)
/home/jfhawkin/.local/lib/python3.12/site-packages/libpysal/weights/contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 3 disconnected components.
  W.__init__(self, neighbors, ids=ids, **kw)
/home/jfhawkin/.local/lib/python3.12/site-packages/libpysal/weights/contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 3 disconnected components.
  W.__init__(self, neighbors, ids=ids, **kw)
/home/jfhawkin/.local/lib/python3.12/site-packages/libpysal/weights/contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 6 disconnected components.
  W.__init__(self, neighbors, ids=ids, **kw)
/home/jfhawkin/.local/lib/python3.12/site-packages/libpysal/weights/contiguity.py:347: UserW

                SF_0           SF_1           SF_2           SF_3  \
count  216623.000000  216623.000000  216623.000000  216623.000000   
mean       -0.000020       0.000005       0.000026      -0.000031   
std         0.039618       0.039618       0.039618       0.039618   
min        -0.703048      -0.647022      -0.605662      -0.649109   
25%        -0.002385      -0.003067      -0.003873      -0.003238   
50%        -0.000217      -0.000362      -0.000502       0.000271   
75%         0.002839       0.002946       0.002958       0.004183   
max         0.693021       0.639795       0.624253       0.659098   

                SF_4  
count  216623.000000  
mean        0.000027  
std         0.039618  
min        -0.631188  
25%        -0.004563  
50%        -0.000455  
75%         0.004220  
max         0.612452  
